In [4]:
# detect camera only and tells you the distance of cell phone to the laptop cam

from ultralytics import YOLO
import cv2

# Load a pretrained YOLO model (adjust path if needed)
model = YOLO("yolov5s.pt")  # Replace with the path to your YOLO model

# Open video capture
cap = cv2.VideoCapture(0)

# Known object parameters for calibration
distance_real = 20  # Known distance to the object in cm
width_real = 16.7     # Known actual width of the object in cm
width_camera = 462  # Perceived width in pixels at the known distance

# Calculate the focal length (calibration step)
focal_r = (distance_real * width_camera) / width_real

while True:
    # Capture frame
    success, frame = cap.read()

    # Check if frame capture is successful
    if not success:
        print("Error: Could not capture frame from camera.")
        break

    # Perform object detection
    results = model(frame, stream=True)

    # Process and display detections
    for r in results:
        boxes = r.boxes
        for bbox in boxes:
            cls_idx = int(bbox.cls[0])
            cls_names = model.names[cls_idx]

            # Calculate the perceived width in pixels
            perceived_width = x2 - x1
            # Calculate the distance
            if perceived_width > 0:  # Avoid division by zero
                distance = (width_real * focal_r) / perceived_width
                distance_text = f"Distance: {distance:.2f} cm"
            else:
                distance_text = "Distance: N/A"

            # Check if the detected object is a "cellphone" 
            if cls_names == "cell phone":  # Adjust the class name as per your model's labels
                x1, y1, x2, y2 = bbox.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 4)
                cv2.putText(frame, f'{cls_names}', (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.putText(frame, distance_text, (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Display the frame with detections
    cv2.imshow('detection', frame)

    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release camera resources and close windows
cap.release()
cv2.destroyAllWindows()
print("Camera closed successfully.")

PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


0: 480x640 1 person, 177.5ms
Speed: 5.5ms preprocess, 177.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 199.0ms
Speed: 6.6ms preprocess, 199.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 174.9ms
Speed: 2.0ms preprocess, 174.9ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 168.0ms
Speed: 3.2ms preprocess, 168.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 couch, 250.3ms
Speed: 2.6ms preprocess, 250.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 173.6ms
Speed: 3.1ms preprocess, 173.6ms inference, 1.0ms postprocess 

In [ ]:
# box with width reading # Perceived width of phone in pixels at the known distance

from ultralytics import YOLO
import cv2

# Load a pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Open video capture
cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        break

    results = model(frame, stream=True)

    for r in results:
        boxes = r.boxes
        for bbox in boxes:
            x1, y1, x2, y2 = bbox.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)  # Convert float to int

            cls_idx = int(bbox.cls[0])
            cls_names = model.names[cls_idx]

            # Calculate the width of the bounding box
            width_cam = x2 - x1

            # Draw bounding box and display width
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 4)
            cv2.putText(frame, f'{cls_names}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Width: {width_cam}px', (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
    # Display the frame
    cv2.imshow('detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [3]:
#final code with real time detection, run with thonny simultaneously to make it work
import socket
import cv2
from ultralytics import YOLO

# Network configuration
HOST = '192.168.129.9'  # Update with your Raspberry Pi's IP
PORT = 65432

# Load a pretrained YOLO model
model = YOLO("yolov5s.pt")

# Open video capture
cap = cv2.VideoCapture(0)

# Calibration parameters
distance_real = 20  # Known distance to the object in cm
width_real = 16.7  # Known actual width of the object in cm
width_camera = 462  # Perceived width in pixels at the known distance
focal_r = (distance_real * width_camera) / width_real  # Focal length

# Font for text overlay
font = cv2.FONT_HERSHEY_SIMPLEX

# Create socket connection
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))

    while True:
        # Capture frame
        success, frame = cap.read()
        if not success:
            print("Error: Could not capture frame from camera.")
            break

        # Perform object detection
        results = model(frame, stream=True)
        detected = False

        # Process and display detections
        for r in results:
            boxes = r.boxes
            for bbox in boxes:
                cls_idx = int(bbox.cls[0])
                cls_names = model.names[cls_idx]

                # Extract bounding box coordinates
                x1, y1, x2, y2 = bbox.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Calculate perceived width
                perceived_width = x2 - x1

                # Check if detected object is a "cell phone"
                if cls_names == "cell phone":
                    detected = True
                    if perceived_width > 0:
                        distance = (width_real * focal_r) / perceived_width
                        distance_text = f"Distance: {distance:.2f} cm"
                        print(f"Detected: {cls_names} at {distance:.2f} cm")

                        # Send signal to Raspberry Pi
                        s.sendall(f"LED_ON:{distance:.2f}".encode())
                    else:
                        print("Error: Invalid perceived width.")

                    # Draw bounding box and distance
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(frame, cls_names, (x1, y1 - 10), font, 0.5, (255, 0, 0), 2)
                    cv2.putText(frame, distance_text, (x1, y2 + 20), font, 0.5, (0, 255, 0), 2)

        # If no cell phone is detected, send an LED_OFF signal
        if not detected:
            s.sendall("LED_OFF".encode())

        # Display the frame
        cv2.imshow('Detection', frame)

        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Release resources
cap.release()
cv2.destroyAllWindows()


PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.




0: 480x640 1 person, 1 chair, 217.7ms
Speed: 5.1ms preprocess, 217.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 2 chairs, 241.1ms
Speed: 4.0ms preprocess, 241.1ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 2 chairs, 211.7ms
Speed: 3.0ms preprocess, 211.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 2 chairs, 212.7ms
Speed: 2.0ms preprocess, 212.7ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 210.5ms
Speed: 4.0ms preprocess, 210.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 chair, 212.3ms
Speed: 2.3ms preprocess, 212.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 chair, 202.3ms
Speed: 1.5ms preprocess, 202.3ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 198.

In [ ]:
#Thonny code
import RPi.GPIO as GPIO
import socket
import time

# GPIO setup
LED_PIN = 17
GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)
GPIO.setup(LED_PIN, GPIO.OUT)
GPIO.output(LED_PIN, GPIO.LOW)  # Start with LED off

# Initialize state variables
led_state = False

# Socket configuration
HOST = '192.168.129.9'  
PORT = 65432

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print("Waiting for connection...")
    conn, addr = s.accept()
    print('Connected by', addr)

    with conn:
        while True:
            data = conn.recv(1024).decode().strip()
            if not data:
                break

            if data.startswith("LED_ON"):
                _, distance_str = data.split(":")
                try:
                    distance = float(distance_str)
                    if distance <= 25:  # Turn on LED for distance <= 25 cm
                        GPIO.output(LED_PIN, GPIO.HIGH)
                        led_state = True
                        print(f"LED turned on. Cell phone detected at {distance:.2f} cm.")
                    else:
                        # Turn off LED if distance > 25 cm
                        GPIO.output(LED_PIN, GPIO.LOW)
                        if led_state:
                            print("LED turned off due to cell phone being too far.")
                            led_state = False
                except ValueError:
                    print("Error: Invalid distance data received.")
            elif data == "LED_OFF":
                GPIO.output(LED_PIN, GPIO.LOW)
                if led_state:
                    print("LED turned off manually.")
                    led_state = False
